In [1]:
!uv pip install datasets pandas numpy matplotlib seaborn pillow tqdm kagglehub

Using Python 3.12.12 environment at: /usr
Audited 8 packages in 249ms


In [2]:
import kagglehub
from datasets import load_dataset
sroie_root   = kagglehub.dataset_download('urbikn/sroie-datasetv2')
fia_root     = kagglehub.dataset_download('nikita2998/find-it-again-dataset')
cord_dataset = load_dataset('naver-clova-ix/cord-v2')

100%|██████████| 834M/834M [00:12<00:00, 70.3MB/s]

Extracting files...


100%|██████████| 643M/643M [00:06<00:00, 111MB/s] 

Extracting files...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

data/validation-00000-of-00001-cc3c5779f(…):   0%|          | 0.00/242M [00:00<?, ?B/s]

data/test-00000-of-00001-9c204eb3f4e1179(…):   0%|          | 0.00/234M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

We will format the 3 datasets under 1 unified format, each record from the datasets will contain:
- `id`:  id, consist of two parts **source_dataset**_**image_file_name_within_soruce** For Cord we will save image in disk
- `source`:   "sroie" | "find_it_again" | "cord"
- `source_id`: filename stem (sroie/fia) or meta.image_id (CORD)
- `split`:"train" | "val" | "test"
- `vendor`: always null for cord since they are blurred and not data in `gt_parse`
- `date`: can also be null for cord
- `total`:
- `is_forged`: applicable only for fia dataset

I know the only required ones are vendor, date, and total. But I am adding them for debugging and the pipeline will only produce the required ones.

In [3]:
# regex

import re

DATE_PATTERN = re.compile(
    r'\b('
    r'\d{4}(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])'              # 20180428  YYYYMMDD
    r'|(?:0[1-9]|[12]\d|3[01])(?:0[1-9]|1[0-2])\d{4}'             # 25032018  DDMMYYYY
    r'|\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}(?=[^\d]|$)'          # 22/03/2018, 21/05/2018TIME
    r'|\d{4}[\/\-\.]\d{1,2}[\/\-\.]\d{1,2}'                        # 2018/03/22
    r'|\d{1,2}[\/\-\.\s][A-Za-z]{3}[\/\-\.\s]\d{2,4}'             # 22 MAR 18, 22/MAR/18
    r'|[A-Za-z]{3}\.?\s+\d{1,2},?\s+\d{4}'                         # OCT 3, 2016
    r')',
    re.IGNORECASE
)


_NUM = r'[\d]{1,3}(?:,\d{3})*(?:\.\d{2})|\d+\.\d{2}'

# works by first match wins
_TOTAL_PATTERNS = [
    # this for the most common format I found while testing the rounded/inclusive total format
    # it will handle typos, new lines numerical total
    re.compile(rf'(?i)(?:rounded|inclus[a-z]*)\b[^\d\n%]*[\r\n]*\s*(?:RM\s*)?({_NUM})(?!\s*%)'),

    # total only on the same line
    re.compile(rf'(?im)^(?!.*(?:%|GST|TAX))(?<!\w)total\b[^\n\d]*?(?:RM\s*)?({_NUM})'),

    # total on the next line
    re.compile(rf'(?im)^(?!.*(?:%|GST|TAX))(?<!\w)total\b[^\n]*\n\s*(?:RM\s*)?({_NUM})'),

    # inclsuive total with RM
    re.compile(rf'(?is)total\s+incl.*?(?:RM\s*)?({_NUM})'),

    # total with AMT post fix
    re.compile(rf'(?i)net\s+amt\b.*?({_NUM})'),
]



In [4]:
# extractors/helpers

import requests
from huggingface_hub import hf_hub_download


# only used as a fallback in the worst cases where cost is very low
# decided to ditch it
_llm = None


def extract_date(ocr_text):
    ocr_text = ocr_text.replace('\r\n', '\n').replace('\r', '\n')
    match = DATE_PATTERN.search(ocr_text)
    return match.group(1).strip() if match else None


def extract_total_regex(text):
    for pattern in _TOTAL_PATTERNS:
        matches = list(pattern.finditer(text))
        if matches:
            # last match
            return matches[-1].group(matches[-1].lastindex)
    return None



def _load_model_gguf():
    global _llm
    if _llm is None:
        model_path = hf_hub_download(
            repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
            filename="qwen2.5-0.5b-instruct-q4_k_m.gguf",
        )
        _llm = Llama(
            model_path=model_path,
            n_ctx=1024,      # context, the receipt text is way lower
            n_threads=4,     # enhances inference on cpu
            verbose=False,
        )


# def extract_total_llm(ocr_text):
#     from llama_cpp import Llama
#     _load_model_gguf()

#     prompt = (
#         "<|im_start|>system\n"
#         "You are a receipt parser. "
#         "Extract ONLY the final total amount paid. "
#         "Reply with the number only, e.g. 12.50. "
#         "If unknown, reply: null\n"
#         "<|im_end|>\n"
#         "<|im_start|>user\n"
#         f"{ocr_text}\n"
#         "<|im_end|>\n"
#         "<|im_start|>assistant\n"
#     )

#     output = _llm(
#         prompt,
#         max_tokens=10,
#         temperature=0,
#         stop=["<|im_end|>", "\n"],
#     )
#     print("called for text:", ocr_text[:50])
#     raw = output["choices"][0]["text"].strip()
#     if raw.lower() == "null":
#         return None
#     try:
#         float(raw)
#         return raw
#     except ValueError:
        return None


def extract_total(ocr_text, record_id):
    # regex first then llm at wors cases
    result = extract_total_regex(ocr_text)
    if result is None:
        # i am not using it actually, though the llm is small and quite fast on cpu
        # but our failures (only fia dataset) are going to be 100 (see last cell)
        # and calling this 100 times introduces latency
        # however in a real setting it is actually quite useful for a handful of failures
        # if you wanna try the following make sure to run `!uv pip install llama-cpp-python` first and uncomment the funciton above
        # result = extract_total_llm(ocr_text)
        pass # for now
    return result

### some test

In [5]:
# let's give our extract total comprehensive test from our datasets knowledge

test_cases = [
    (14.20, """PASARAYA CINWA SDN BHD
(1113440A)
NO.6,8, 10& 12 JALAN PERMAS 4/3
BANDAR BARU PERMAS JAYA,
81750 JOHOR BAHRU,JOHOR.
TEL : 07-3881722
FAX : 07-3873971
GST ID : 000931172352
TAX INVOICE
DOC NO.
CASHIER
SALESPERSON
: CS01076695
: JULIANA
: JULIANA
DATE: 15/06/2018
TIME: 09:40:00
REF.:
ITEM
QTY
S/PRICE
(GST)
(GST)
S/PRICE
AMOUNT
TAX
9557821042500
YELLOW ROCK SUGAR 250G
2200056
GREEN BEAN
280G
9556894200008
ROYAL UMBRELLA BERAS WANGI 1KG
1
1
1
2.50
2.80
8.90
2.50
2.80
8.90
2.50
2.80
8.90
SR0
SR0
SR0
TOTAL QTY:
3
14.20
TOTAL SALES (EXCLUDING GST) :
14.20
DISCOUNT :
0.00
TOTAL GST :
0.00
ROUNDING :
0.00
TOTAL SALES (INCLUSIVE OF GST) :
14.20
CASH :
20.20
CHANGE :
6.00
GST SUMMARY
TAX CODE
SR0
%
0
AMT (RM)
14.20
TAX (RM)
0.00
TOTAL :
14.20
0.00
GOODS SOLD ARE NOT RETURNABLE, THANK YOU
** RE-PRINT **
"""),
    (30.45, """PASAR RAYA MEGA MAJU
(SEMENYIH) SDN BHD
(767840-W)
NO 24, 25 & 26,
JALAN PUSAT PERNIAGAAN BUNGA RAYA 4,
PUSAT PERNIAGAAN BUNGA RAYA,
TEL:
FAX:
GST REG NO: 000746659840
INVOICE NO
: 01-283886
DATE
: 03/07/2018 5:22:43 PM
CASHIER
: SITI
TAX INVOICE
DESCRIPTION
QTY
PRICE
AMOUNT
1
2
SRO 9555544201075
PRO BALANCE 400GM - BEFF
SR0 9555544201198
PRO BALANCE LAMB 400GM
4
3
4.35
4.35
17.40
13.05
TOTAL:
30.45
DISCOUNT:
0.00
TOTAL SALES INCLUSIVE GST @0.00%:
30.45
CASH RECEIVED :
30.50
CHANGE :
0.05
GST SUMMARY
%
AMOUNT(RM)
TAX(RM)
SR0
0.00
30.45
0.00
THANK YOU. PLEASE COME AGAIN
TERIMA KASIH. SILA DATANG LAGI
GOOD SOLD ARE NOT REFUNABLE AND RETURNABLE
"""),
    (11.32, """3180303
LIAN HING STATIONERY SDN BHD
(162761-M)
NO.32 & 33, JALAN SR 1/9, SEKSYEN 9,
TAMAN SERDANG RAYA,
43300 SERI KEMBANGAN, SELANGOR
DARUL EHSAN
GST ID : 002139201536
TAX INVOICE
27/03/2018
NO : CS-20243
QTY
TAX
RM
F/CASTELL 187057- 75 TACK-LT
2
SR
12.00
75G- WHITE (NEW) @ 5.6600
TOTAL AMT INCL. GST @ 6% :
12.00
ROUNDING ADJUSTMENT :
TOTAL AMT PAYABLE :
12.00
PAID AMOUNT :
20.00
CHANGE :
8.00
TOTAL QTY TENDER :
2
GST SUMMARY
AMOUNT
TAX
(RM)
(RM)
SR @ A
11.32
0.68
TOTAL
11.32
0.68
THANK YOU
FOR ANY ENQUIRY, PLEASE CONTACT US:
"""),
    (24.63, """AEON CO. (M) BHD (126926-H)
3RD FLR, AEON TAMAN MALURI SC
JLN JEJAKA, TAMAN MALURI
CHERAS, 55100 KUALA LUMPUR
GST ID : 002017394688
SHOPPING HOURS
SUN-THU:1000 HRS - 2200 HRS
FRI-SAT:1000 HRS - 2300 HRS
1X 000006142384
7.51SR
MUNCHY`S CREAM
3X 000005709410
18.60SR
TOPVALU BESTPRI @6.20
SUB-TOTAL
26.11
TOTAL SALES INCL GST
26.11
ROUNDING ADJ
-0.01
TOTAL AFTER ADJ INCL GST
26.10
CASH
50.00
ITEM COUNT 4
CHANGE AMT
23.90
INVOICE NO: 2018042810080010325
GST SUMMARY
AMOUNT
TAX
SR @ 6%
24.63
1.48
TOTAL
24.63
1.48
28/04/2018 15:35
1008 001 0010325
0304662 HEMADAS A/L BALOO
AEON BANDAR PUCHONG
TEL 1-300-80-AEON (2366)
THANK YOU FOR YOUR PATRONAGE
PLEASE COME AGAIN
"""),
    (26.70, """MR. D.I.Y. (M) SDN BHD
CO-REG:860671-D
LOT 1851-A & 1851-B, JALAN KPB 6,
KAWASAN PERINDUSTRIAN BALAKONG,
43300 SERI KEMBANGAN, SELANGOR
(GST ID NO :000306020352)
(SELAYANG MALL)
-TAX INVOICE-
WASH MITT 402
JD62/3 - 6/240
9064038
STICKER WS-SB
PB21 - 20/800
9013039
EMERGENCY LIGHT LED716
NE51/2/3 - 60
6939020440166
ITEM(S) : 3
1 X
1 X
1 X 16.90
*S
*S
*S
QTY(S) : 3
TOTAL INCL. GST@6%
CASH
CHANGE
GST @6% INCLUDED IN TOTAL
RM 26.70
RM 50.00
RM 23.30
RM 1.51
06-05-16 17:11 SH01 ZJ08
OPERATOR SLC - EALIL ARASI
EXCHANGE ARE ALLOWED WITHIN
3 DAY WITH RECEIPT.
STRICTLY NO CASH REFUND.
4.30
4.30
5.50
5.50
16.90
T2 R000602051
"""),
    (7.30, """SIN LIANHAP SDN BHD
LOT 13, JALAN
SIN LIANHAP SDN BHD
LOT 13, JALAN IPOH,
KG BATU 30 , ULU YAM LAMA
44300 BTG KALI, SELANGOR.
TEL:03-60752222(HUNTING LINE) FAX: 03-60752572
(COMPANY REG NO: 284922-D)
(GST REG NO: 001610833920)
TAX INVOICE
CASH CUSTOMER
INVOLCE NO.:
:H0003939
DATE:
:05/02/2018
CASHIER#:
:
RM
CODE
PPD 4MM DENLIME
1.887
SR
1.000PC
2.00
6023# GARDEN
5.000
SR
1.000 PC
5.30
SUBTOTAL :
7.30
TOTAL SALES EXCLUDING GST :
6.69
TOTAL SALES INCLUSLVE OF GST:
7.30
ROUNDING ADJUSTMENT :
0.00
PAYMENT :
7.30
CHANGE DUE :
0.00
TOTAL ITEM(S) :
GST SUMMARY
SR
AMONUT
6.89
TAX
0.41
THANK YOU
PLEASE COME AGAIN
X
X
2
@6

"""),
    (112.35, """
    ONE ONE THREE SEAFOOD RESTAURANT SDN BHD
(1120908-M)
NO.1, TAMAN SRI DENGKIL, JALAN AIR HITAM
43800 DENGKIL, SELANGOR.
(GST REG. NO : 000670224384)
TAX INVOICE
TABLE 03
BILL#:V001-539233
ORDER#: 139336
DATE
: 23-06-2018 23:28:57
CASHIER: 113 CASHIER
PAX(S):
0
QTY
DESCRIPTION
TOTAL TAX
1
D
70.00 SR
GROUPER
1
D
15.00 SR
VEGE ITEM
1
D
8.00 SR
OMELLETE ITEM
1
D
8.00 SR
WHITE RICE
1
D
5.00 SR
BEVERAGE
TOTAL (EXCLUDING GST):
106.00
GST PAYABLE:
6.36
TOTAL (INCLUSIVE OF GST):
112.36
ROUNDING ADJ:
-0.01
TOTAL :
112.35
CLOSED: 1
28-05-2018
23:50:00
SERVER: 113 CASHIER
CASH :
112.35
GST SUMMARY
AMOUNT(RM)
TAX(RM)
SR
(@ 6%)
106.00
6.36
**** THANK YOU ****
PLEASE COME AGAIN

    """),
    (74.20, """SYARIKAT PERNIAGAAN GIN KEE
(81109-A)
NO 290. JALAN AIR PANAS.
SETAPAK.
53200, KUALA LUMPUR.
TEL : 03-40210276
GST ID : 000750673920
SIMPLIFIED TAX INVOICE
CASH
DOC NO.
: CS00012922
DATE: 24/01/2018
CASHIER
: USER
TIME: 10:50:00
SALESPERSON :
REF.:
ITEM
QTY
S/PRICE
AMOUNT
TAX
1700
PASIR HALUS (BAG)
2430
CEMENT (50KG)
6
6.36
38.16
SR
2
18.02
36.04
SR
TOTAL QTY:
8
74.20
TOTAL SALES (EXCLUDING GST) :
70.00
DISCOUNT :
0.00
TOTAL GST :
4.20
ROUNDING :
0.00
TOTAL SALES (INCLUSIVE OF GST) :
74.20
CASH :
74.20
CHANGE :
0.00
GST SUMMARY
TAX CODE
%
AMT(RM)
TAX (RM)
SR
6
70.00
4.20
TOTAL :
70.00
4.20
GOODS SOLD ARE NOT RETURNABLE, THANK YOU.
"""),
    (346.51, """TAX INVOICE
GREAT ZONE HOUSEHOLD CENTRE SDN BHD
(801049-U)
60 & 62. JALAN CIKU,
86000,KLUANG
GST REG.: 001833082880
DOCUMENT NO. :
DATE :
DEBTOR :
MEMBER :
TERMINAL :
CASHIER :
KLG0201130227
18/02/2018 06:08:59 PM
KLG0201
KLG2
DESC
U.PRICE
DISC
AMOUNT
TAX
QTY
RM
RM
CODE
200SHEET COMPACT SERVIETTES (NTE)
1 PC
*
3.30
0.00
3.30 SR
1.5" RUBBER BAND
1 200G *
3.60
0.00
3.60 SR
PP 3X5 04
1 200G *
2.50
0.00
2.50 SR
18MMX90Y OPP TAPE
1 PC
*
1.60
0.00
1.60 SR
911 TAPE DISPENSER
1 PC
*
8.60
0.00
8.60 SR
PSI-890 PENSONIC DRY IRON WITH SPRAY
1 PC
*
49.90
0.00
49.90 SR
AP-17 APRON PLAIN CLOTH
20 PC
*
10.90
0.00
218.00 SR
TALI UKURAN
1 PC
*
1.00
0.00
1.00 SR
CC-100 100PC CITE CANDLE
1 PC
*
22.90
0.00
22.90 SR
F1502 SCISSOR 08C (CH) (TLC)
6 PC
*
6.00
0.00
36.00 SR
2KG HC KITCHEN SCALE
1 PC
*
19.90
0.00
19.90 SR
SUB TATAL (EXCLUSIVE GST) :
GST 6% :
SERVICE CHARGE :
ROUNDING ADJUSTMENT :
ROUNDED TOTAL (RM):
346.51
20.79
0.00
0.00
367.30
CREDIT CARD
2367.30
GST SUMMARY
AMOUNT(RM)
TAX(RM)
SR @ 6%
346.51
20.79
"""),
    (10.50, """LIM SENG THO HARDWARE TRADING
NO 7. SIMPANG OFF BATU VILLAGE.,
JALAN IPOH BATU 5, 51200 KUALA LUMPUR.
MALAYSIA
TEL & FAX NO: 03-6258 7191
03-6258 7191
COMPANY REG NO.: (002231061-T)
GST REG NO.: 001269075968
TAX INVOICE
INVOICE NO.:
CS 24358
DATE:
28/02/2018 12:42
CASHIER#:
LST
RM
CODE
BEG GUNI
15.00 NOS
X
0.70
10.50
SR
SUBTOTAL:
10.50
TOTAL INCL OF GST
10.50
PAYMENT:
10.50
CHANGE DUE:
0.00
TOTAL ITEM(S): 15
GST SUMMARY
AMOUNT(RM)
TAX(RM)
SR
@ 6%
9.91
0.59
THANK YOU
PLEASE COME AGAIN
*GOODS SOLD ARE NOT RETURNABLE*
""")
]

print(f"{'EXPECTED':<25} | {'EXTRACTED':<18} | {'PASSED'}")
print("-" * 65)

for expected, text in test_cases:
    found = False
    total = float(extract_total_regex(text))
    print(f"{expected:<25} | {total:<18} | {total==expected}")


EXPECTED                  | EXTRACTED          | PASSED
-----------------------------------------------------------------
14.2                      | 14.2               | True
30.45                     | 30.45              | True
11.32                     | 11.32              | True
24.63                     | 24.63              | True
26.7                      | 26.7               | True
7.3                       | 7.3                | True
112.35                    | 112.36             | False
74.2                      | 74.2               | True
346.51                    | 346.51             | True
10.5                      | 10.5               | True


### .

In [6]:
import json
from pathlib import Path
import pandas as pd


def load_sroie(sroie_root):
    records = []
    sroie_root = Path(sroie_root)

    for split in ['train', 'test']:
        entities_dir = sroie_root / "SROIE2019"/ split / 'entities'

        for entity_file in sorted(entities_dir.glob('*.txt')):
            source_id = entity_file.stem

            with open(entity_file, 'r', encoding='utf-8') as f:
                labels = json.load(f)

            records.append({
                'id':         f'sroie_{source_id}',
                'source':     'sroie',
                'source_id':  source_id,
                'split':      split,
                'vendor':     labels.get('company'),
                'date':       labels.get('date'),
                'total':      labels.get('total'),
                'is_forged':  0,
            })

    return records


def load_find_it_again(fia_root):
    records = []
    fia_root = Path(fia_root) / 'findit2'

    for split in ['train', 'val', 'test']:
        label_file = fia_root / f'{split}.txt'
        img_dir    = fia_root / split

        if not label_file.exists():
            continue

        df = pd.read_csv(label_file)

        for _, row in df.iterrows():
            img_filename = row['image']
            source_id    = Path(img_filename).stem
            is_forged    = int(row['forged'])

            ocr_file = img_dir / f'{source_id}.txt'
            img_file = img_dir / img_filename

            # drop entries where neither image nor OCR exist on disk
            if not ocr_file.exists() and not img_file.exists():
                continue

            ocr_text = None
            vendor   = None
            if ocr_file.exists():
                with open(ocr_file, 'r', encoding='utf-8') as f:
                    ocr_lines = f.read().splitlines()
                ocr_text = '\n'.join(ocr_lines)
                vendor   = ocr_lines[0].strip() if ocr_lines else None

            record = {
                'id': f'fia_{source_id}',
                'source': 'find_it_again',
                'source_id': source_id,
                'split': split,
                'vendor': vendor,
                'date': extract_date(ocr_text) if ocr_text else None,
                'is_forged': is_forged,
            }
            record['total'] = extract_total(ocr_text, record) if ocr_text else None
            records.append(record)

    return records


def load_cord(cord_dataset):

    # cord has lots of anomalies in their total
    # this handle eerything look at 'extract_cord_total'

    # we got to take rid of RB RM . , etc
    def parse_amount(s):

      # if list, take first element
      if isinstance(s, list):
        s = s[0] if s else None
      if not s or not isinstance(s, str):
        return None

      # stripping currency symbols, spaces, letters (Rp, RM, etc.)
      cleaned = re.sub(r'[^\d.,]', '', s.strip())
      cleaned = cleaned[1:] if cleaned.startswith(".") or cleaned.startswith(
        ",") else cleaned  # remove the remeaining garabage
      cleaned = cleaned[:-1] if cleaned.endswith(".") or cleaned.endswith(",") else cleaned  # here again
      if not cleaned:
        return None

      # removing dots multiple
      if re.match(r'^\d{1,3}(\.\d{3})+$', cleaned):
        cleaned = cleaned.replace('.', '')
      # single dots
      elif re.match(r'^\d+\.\d{3}$', cleaned):
        cleaned = cleaned.replace('.', '')
      # 1,220,500 or 1,250.00
      else:
        cleaned = cleaned.replace(',', '')

      try:
        return float(cleaned)
      except ValueError:
        return None

    def extract_cord_total(total_obj, menu):

        # extract total by priority and handle weird occurance of total

        # 'total_price' is always first
        val = parse_amount(total_obj.get('total_price') if isinstance(total_obj, dict) else None)
        if val is not None:
            return str(int(val)) if val == int(val) else str(val)

        # during testing we found out that if the 'total_price' is absent and 'menu' is a dict (single items) not a list
        # then we can extract thet total from there which is the price of the single item. I got to know this after having CORD in failure_dataset (see last cell)
        if isinstance(menu, dict):
            val = parse_amount(menu.get('price'))
            if val is not None:
                return str(int(val)) if val == int(val) else str(val)

        # aight no 'total_price' no menu dict then try these
        # only'cashprice'? subtract from change if found and return it
        # only'creditcardprice'? subtract from change if found and return it
        # both? add then subtract from change if found and return
        if total_obj.get('cashprice', None) or total_obj.get('creditcardprice', None) or total_obj.get("changeprice", None):
            cash = parse_amount(total_obj.get('cashprice')) or 0
            credit = parse_amount(total_obj.get('creditcardprice')) or 0

            # sometimes the changeprice is a list? and cash, credt, total are absent which is very weird
            change_price = 0
            try:
                change_price_obj = total_obj.get('changeprice')
                if isinstance(change_price_obj, list):
                    parsed = [parse_amount(x) for x in change_price_obj if parse_amount(x) is not None]
                    change_price = max(parsed) if parsed else 0 # i am just gonna get the max element
                elif isinstance(change_price_obj, str):
                    change_price = parse_amount(change_price_obj) or 0
            except:
              pass
            total = cash + credit - change_price # we subtract from change price if found
            return str(total)

        return None

    records = []

    split_map = {'train': 'train', 'validation': 'val', 'test': 'test'}

    for hf_split, split in split_map.items():
        if hf_split not in cord_dataset:
            continue

        for entry in cord_dataset[hf_split]:
            gt        = json.loads(entry['ground_truth'])
            meta      = gt.get('meta', {})
            source_id = str(meta.get('image_id', ''))


            gt_parse  = gt.get('gt_parse', {})
            total_obj = gt_parse.get('total', {}) or {}
            menu      = gt_parse.get('menu', [])
            total     = extract_cord_total(total_obj, menu)

            records.append({
                'id':         f'cord_{source_id}',
                'source':     'cord',
                'source_id':  source_id,
                'split':      split,
                'vendor':     None, # blurred receipt no vendor
                'date':       None, # same
                'total':      total,
                'is_forged':  0,
            })

    return records

In [7]:
def build_unified_dataset(
    sroie_root,
    fia_root,
    cord_dataset
):
    records = []
    records += load_sroie(sroie_root)
    records += load_find_it_again(fia_root)
    records += load_cord(cord_dataset)

    df = pd.DataFrame(records, columns=[
        'id', 'source', 'source_id', 'split',
        'vendor', 'date', 'total', 'is_forged'
    ])

    return df

df = build_unified_dataset(sroie_root, fia_root, cord_dataset)


print('\nSample row 1:')
print(df.iloc[0].to_dict())
print('\nSample row 2:')
print(df.iloc[len(df)//2].to_dict())
print('\nSample row 3:')
print(df.iloc[len(df)-1].to_dict())


Sample row 1:
{'id': 'sroie_X00016469612', 'source': 'sroie', 'source_id': 'X00016469612', 'split': 'train', 'vendor': 'BOOK TA .K (TAMAN DAYA) SDN BHD', 'date': '25/12/2018', 'total': '9.00', 'is_forged': 0}

Sample row 2:
{'id': 'fia_X51007846350', 'source': 'find_it_again', 'source_id': 'X51007846350', 'split': 'train', 'vendor': 'PASARAYA CINWA SDN BHD', 'date': '15/06/2018', 'total': '14.20', 'is_forged': 0}

Sample row 3:
{'id': 'cord_99', 'source': 'cord', 'source_id': '99', 'split': 'test', 'vendor': None, 'date': None, 'total': '75000', 'is_forged': 0}


In [8]:
import math

def get_failures(df):
    failures = []

    # all should have total
    failed = df[df['total'].isnull()].copy()
    failed['fail_type'] = 'total'
    failures.append(failed)

    # vendor + date: CORD null by design, excluding it
    for field in ['vendor', 'date']:
        failed = df[(df['source'] != 'cord') & df[field].isnull()].copy()
        failed['fail_type'] = field
        failures.append(failed)

    return pd.concat(failures, ignore_index=True)


def save_failed_ocrs(failed_df, out_path = 'failed_ocrs.jsonl'):
    with open(out_path, 'w', encoding='utf-8') as f:
        for _, row in failed_df.iterrows():
            record = row.to_dict()
            # clean NaN values
            record = {k: (None if isinstance(v, float) and math.isnan(v) else v) for k, v in record.items()}
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    print(f"Saved {len(failed_df)} failed records to {out_path}")

def summarize_failures(failed_df):
    print("\nFailure summary:")
    print(failed_df.groupby(['fail_type', 'source', 'split']).size().to_string())
    print(f"\nTotal failure records: {len(failed_df)}")
    print(f"Unique receipts with at least one failure: {failed_df['id'].nunique()}")


failed_df = get_failures(df)
summarize_failures(failed_df)
save_failed_ocrs(failed_df, 'failed_ocrs.jsonl')


Failure summary:
fail_type  source         split
date       find_it_again  val        1
total      find_it_again  test      56
                          train    131
                          val       52

Total failure records: 240
Unique receipts with at least one failure: 240
Saved 240 failed records to failed_ocrs.jsonl


The required output

In [9]:
import math

def _is_null(val) -> bool:
    if val is None:
        return True
    if isinstance(val, float) and math.isnan(val):
        return True
    return False

out_path = 'unified_corpus.jsonl'
# removing the 100 failed (fia only) from the actual data, why? see last cell
failed_ids = set(failed_df['id'])
clean_df = df[~df['id'].isin(failed_ids)]

with open(out_path, 'w', encoding='utf-8') as f:
    for _, row in clean_df.iterrows():
        record = {
            'id':        row['id'],
            'vendor':    row['vendor'] if not _is_null(row['vendor']) else None,
            'date':      row['date']   if not _is_null(row['date'])   else None,
            'total':     row['total']  if not _is_null(row['total'])  else None,
            'is_forged': 0,
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"Saved {len(clean_df)} predictions to {out_path} ({len(failed_ids)} dropped)")

Saved 2720 predictions to unified_corpus.jsonl (240 dropped)


### failures
Actually the 240 failed are all from Find-it-Again. Becuase it is largely malformed and has a lot of anomalies. However can be fixed by finding patterns like:
- 'AMOUNT DUE 30.25' on the same line instead of total
- 'TL:RM' or 'TL: RM' then next line total number  
- Mapping total to correct number if it was found in a series of strings followed by exact same number of series of number like:

    SUB TOTAL
    
    ROUNDING ADJ.
    
    TOTAL (INCL.GST) ---------------> (3rd item in string series)
    
    CASH (MYR)
    
    CASH CHANGE:
    
    21.50
    
    0
    
    21.50 ----------> Actual total (3rd item in number series)
    
    22.00
    
    0.50


And much more by studying the failures in the following commented function

If you are still seeing the following comment then, I had no time to do it :(



> Because I am short of time for CodeStacker (I still have 2 more parts) then I will skip catching every possible failure of FIA dataset. However CORD and SROIE are fine. I kind of mitigated this in the next part via better regex, though still not 100% perfect.



In [10]:


# import matplotlib.pyplot as plt
# import matplotlib.image as mpimg
# from PIL import Image

# def show_receipt(row):
#     fig, axes = plt.subplots(1, 2, figsize=(14, 10))

#     try:
#         img = Image.open(row['image_path'])
#         axes[0].imshow(img)
#         axes[0].axis('off')
#         axes[0].set_title(f"{row['id']} | {row['source']} | {row['split']}", fontsize=9)
#     except Exception as e:
#         axes[0].text(0.5, 0.5, f"Image not found\n{e}", ha='center', va='center')
#         axes[0].axis('off')
#     axes[1].axis('off')
#     ocr = row['ocr_text'] or '(no OCR text)'
#     axes[1].text(0, 1, ocr, va='top', ha='left', fontsize=7, family='monospace',
#                  wrap=True, transform=axes[1].transAxes)
#     axes[1].set_title(f"total={row['total']}  date={row['date']}  vendor={row['vendor']}", fontsize=9)

#     plt.tight_layout()
#     plt.show()


# for _, row in failed_df.iterrows():
#     show_receipt(row)


###.